# Week 12: Distill ViT with a **pretrained** CNN teacher (no teacher training)

Loads **ResNet-56** weights pretrained on CIFAR-10 via `torch.hub` (`chenyaofo/pytorch-cifar-models`), freezes the teacher, and trains a ViT student with **CE + KL distillation**.

**Speed-oriented defaults (tune in Config):**
- **Larger batch** (`BATCH_SIZE=2048`) on H100-class GPUs
- **LR** = `1e-3 × (batch / 512)` — **linear scaling only** from the known-good `1e-3 @ 512` recipe (do **not** use an extra 3× multiplier; that stalled ~47% accuracy).
- **`T` and `α`** default back to stable distillation (`T=3`, mild optional α schedule)
- **Teacher forward in BF16 autocast** (same as student)
- **Sparse validation:** every 2 epochs until test ≥ `VALIDATE_DENSE_AFTER`, then every epoch (so we do not miss the 80% crossing)
- **`cudnn.benchmark=True`** on CUDA

**Metric:** wall-clock seconds to **80%** test accuracy (edit `TARGET_ACC` in the config cell).

## Setup

In [ ]:
import sys
import math
import time
from pathlib import Path
from typing import Dict, Any, List, Optional

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model import count_parameters
from src.utils import (
    get_device,
    get_cifar10_loaders,
    get_model,
    set_seed,
    validate,
    get_peak_gpu_memory_mb,
    reset_peak_gpu_memory,
)

print(torch.__version__, get_device())

## Config

In [ ]:
SEED = 42
TARGET_ACC = 80.0
MAX_EPOCHS = 100

# --- Throughput: larger batch; LR only linear in batch vs 1e-3 @ 512 (no extra multiplier) ---
BATCH_SIZE = 2048
LR_REF_BATCH = 512
LR_REF = 1e-3            # LR that worked @ 512 in the ~288s / 80% baseline
LR = LR_REF * (BATCH_SIZE / LR_REF_BATCH)   # 4e-3 @ 2048

NUM_WORKERS = 4

WARMUP_EPOCHS = 5

# --- Distillation (conservative defaults after LR/α/T were too aggressive) ---
DISTILL_TEMP = 3.0
# Mild α schedule: set START==END for fixed α=0.5 (original behavior)
ALPHA_START = 0.55
ALPHA_END = 0.45
ALPHA_ANNEAL_LAST_EP = 12

# --- Validation: cheap until we are close to the target bar ---
VALIDATE_EVERY_COARSE = 2   # validate every N epochs while test < VALIDATE_DENSE_AFTER
VALIDATE_DENSE_AFTER = 74.0 # then validate every epoch so we do not miss TARGET_ACC

set_seed(SEED, deterministic=False)
print(
    f"batch={BATCH_SIZE} LR={LR:.2e} T={DISTILL_TEMP} α {ALPHA_START}→{ALPHA_END} by ep{ALPHA_ANNEAL_LAST_EP} | "
    f"val every {VALIDATE_EVERY_COARSE} ep until test≥{VALIDATE_DENSE_AFTER}%"
)

## Data

In [ ]:
train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment_train=True,
    use_randaugment=True,
    randaugment_num_ops=2,
    randaugment_magnitude=9,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2,
)
xb, _ = next(iter(train_loader))
device = get_device()
print(len(train_loader), "batches", xb.shape)

## Frozen pretrained CNN teacher

In [ ]:
device = get_device()
cnn_teacher = torch.hub.load(
    "chenyaofo/pytorch-cifar-models",
    "cifar10_resnet56",
    pretrained=True,
)
cnn_teacher.eval()
for p in cnn_teacher.parameters():
    p.requires_grad_(False)
cnn_teacher = cnn_teacher.to(device)
_, acc = validate(cnn_teacher, test_loader, nn.CrossEntropyLoss(), device)
print(f"Teacher test acc: {acc:.2f}% | params: {count_parameters(cnn_teacher):,}")

## ViT student

In [ ]:
def make_student():
    return get_model(patch_size=4, embed_dim=192, depth=6, num_heads=6, dropout=0.0)

student = make_student().to(device)
print(count_parameters(student), "params")

## Train (distillation)

In [ ]:
def alpha_for_epoch(ep: int) -> float:
    """Linearly anneal KL weight α from ALPHA_START → ALPHA_END by ALPHA_ANNEAL_LAST_EP."""
    last = max(2, int(ALPHA_ANNEAL_LAST_EP))
    if ep >= last:
        return ALPHA_END
    u = (ep - 1) / max(1, last - 1)
    return ALPHA_START + (ALPHA_END - ALPHA_START) * u


def distill_train(student, teacher, train_loader, test_loader):
    device = get_device()
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    student = student.to(device)
    teacher = teacher.to(device)
    if device.type == "cuda":
        student = torch.compile(student, mode="reduce-overhead")

    opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

    def lr_lambda(ep):
        if ep < WARMUP_EPOCHS:
            return (ep + 1) / WARMUP_EPOCHS
        p = (ep - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    T = DISTILL_TEMP

    hist: Dict[str, List] = {"test_acc": [], "train_loss": [], "wall_time": [], "alpha": []}
    best, ttt = 0.0, None
    last_val_acc: Optional[float] = None
    t0 = time.perf_counter()

    for ep in range(1, MAX_EPOCHS + 1):
        alpha = alpha_for_epoch(ep)
        student.train()
        tot = 0.0
        t_ep = time.perf_counter()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            if use_amp:
                with torch.amp.autocast(device_type="cuda", dtype=dtype):
                    zs = student(x)
                with torch.no_grad():
                    with torch.amp.autocast(device_type="cuda", dtype=dtype):
                        zt = teacher(x)
            else:
                zs = student(x)
                with torch.no_grad():
                    zt = teacher(x)
            ce = F.cross_entropy(zs, y)
            kl = F.kl_div(
                F.log_softmax(zs / T, dim=-1),
                F.softmax(zt / T, dim=-1),
                reduction="batchmean",
            ) * (T ** 2)
            loss = (1 - alpha) * ce + alpha * kl
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()
            tot += loss.item()
        sched.step()
        wt = time.perf_counter() - t_ep

        run_val = (ep == 1) or (ep == MAX_EPOCHS)
        if last_val_acc is not None and last_val_acc >= VALIDATE_DENSE_AFTER:
            run_val = True
        elif ep % VALIDATE_EVERY_COARSE == 0:
            run_val = True
        if run_val:
            _, acc = validate(student, test_loader, nn.CrossEntropyLoss(), device)
            last_val_acc = acc
        else:
            acc = last_val_acc if last_val_acc is not None else 0.0

        best = max(best, acc)
        if ttt is None and acc >= TARGET_ACC:
            ttt = time.perf_counter() - t0
        hist["train_loss"].append(tot / len(train_loader))
        hist["test_acc"].append(acc)
        hist["wall_time"].append(wt)
        hist["alpha"].append(alpha)
        tag = "~" if not run_val else ""
        print(
            f"ep{ep:3d} α={alpha:.2f} loss {hist['train_loss'][-1]:.4f} "
            f"test{tag} {acc:.2f}% ({wt:.1f}s/ep)"
        )
        if ttt is not None:
            print(f"  >>> reached {TARGET_ACC}% test accuracy in {ttt:.2f}s wall-clock")
            break
    return {"history": hist, "best_acc": best, "time_to_target": ttt, "total_time": time.perf_counter() - t0}


set_seed(SEED)
reset_peak_gpu_memory()
student = make_student()
results = distill_train(student, cnn_teacher, train_loader, test_loader)
print("\n--- done ---")
print(f"time to {TARGET_ACC}%: {results['time_to_target']}")
print(f"best acc: {results['best_acc']:.2f}%  total: {results['total_time']:.1f}s")
print("peak MB:", get_peak_gpu_memory_mb())

## Plots

In [ ]:
import matplotlib.pyplot as plt

h = results["history"]
cum = [sum(h["wall_time"][: i + 1]) for i in range(len(h["wall_time"]))]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h["test_acc"], color="#0077b6", lw=2)
ax[0].axhline(TARGET_ACC, color="gray", ls="--")
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("test %")
ax[0].set_title("Test accuracy")
ax[0].grid(alpha=0.3)
ax[1].plot(cum, h["test_acc"], color="#0077b6", lw=2)
ax[1].axhline(TARGET_ACC, color="gray", ls="--")
if results["time_to_target"]:
    ax[1].axvline(results["time_to_target"], color="red", ls=":", label="hit")
    ax[1].legend()
ax[1].set_xlabel("wall time (s)")
ax[1].set_ylabel("test %")
ax[1].set_title("vs wall clock")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "week11_results.png", dpi=150, bbox_inches="tight")
plt.show()